# Load Saved Models and Run Inference

Run this after the training notebook has saved the `.pkl` files. It loads the stored models and applies them to the saved test data.

In [ ]:
import os
import glob
import io
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 42
MODEL_DIR = Path("/content/ml_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_DATASET_FILE_ID = "1sEynbX9BezEIS4MBdbY2jWsgkii4uJON"
DRIVE_MODEL_FOLDER_ID = "1elG38b6F7Ptfq2spqnW5zAxzqgPR4vDh"

In [ ]:
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

expected_paths = [
    "/content/drive/MyDrive/ByteSmart Internship - Ranveer Singh/cold_source_control_dataset.csv",
]

csv_path = None
for candidate in expected_paths:
    if os.path.exists(candidate):
        csv_path = candidate
        break

if csv_path is None:
    matches = glob.glob(
        "/content/drive/**/ByteSmart Internship - Ranveer Singh/cold_source_control_dataset.csv",
        recursive=True,
    )
    if not matches:
        print("Could not find the CSV in mounted Drive paths. Downloading it by Drive file ID instead.")
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload

        auth.authenticate_user()
        drive_service = build("drive", "v3")
        request = drive_service.files().get_media(fileId=DRIVE_DATASET_FILE_ID)
        csv_path = "/content/cold_source_control_dataset.csv"
        with open(csv_path, "wb") as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
                if status:
                    print(f"Download progress: {int(status.progress() * 100)}%")
    else:
        csv_path = matches[0]

df = pd.read_csv(csv_path)
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df = df.sort_values("Timestamp").reset_index(drop=True)

rename_map = {
    "Inlet_Temperature(°C)": "Inlet_Temperature",
    "Outlet_Temperature(°C)": "Outlet_Temperature",
    "Ambient_Temperature(°C)": "Ambient_Temperature",
    "Cooling_Unit_Power_Consumption(kW)": "Cooling_Power_kW",
    "Chiller_Usage(%)": "Chiller_Usage",
    "AHU_Usage(%)": "AHU_Usage",
    "Total_Energy_Cost($)": "Total_Energy_Cost",
    "Temperature_Deviation(°C)": "Temperature_Deviation",
    "Server_Workload(%)": "Server_Workload",
}
df = df.rename(columns=rename_map)
df["Thermal_Delta"] = df["Outlet_Temperature"] - df["Inlet_Temperature"]
df["Increase_Chiller_Label"] = (df["Cooling_Strategy_Action"] == "Increase Chiller").astype(int)

feature_cols = [
    "Server_Workload",
    "Inlet_Temperature",
    "Outlet_Temperature",
    "Ambient_Temperature",
    "Chiller_Usage",
    "AHU_Usage",
    "Temperature_Deviation",
    "Thermal_Delta",
]
regression_target = "Cooling_Power_kW"
classification_target = "Increase_Chiller_Label"
cluster_cols = feature_cols + ["Cooling_Power_kW", "Total_Energy_Cost"]

print("Dataset:", csv_path)
print("Shape:", df.shape)
display(df.head())

In [ ]:
sample_df, remainder_df = train_test_split(
    df,
    train_size=0.80,
    random_state=RANDOM_STATE,
    stratify=df["Cooling_Strategy_Action"],
)

coverage_rows = []
numeric_cols = [
    "Server_Workload",
    "Inlet_Temperature",
    "Outlet_Temperature",
    "Ambient_Temperature",
    "Cooling_Power_kW",
    "Chiller_Usage",
    "AHU_Usage",
    "Total_Energy_Cost",
    "Temperature_Deviation",
]

for col in numeric_cols:
    coverage_rows.append({
        "column": col,
        "full_min": df[col].min(),
        "sample_min": sample_df[col].min(),
        "full_max": df[col].max(),
        "sample_max": sample_df[col].max(),
        "full_mean": df[col].mean(),
        "sample_mean": sample_df[col].mean(),
        "mean_difference": sample_df[col].mean() - df[col].mean(),
    })

coverage = pd.DataFrame(coverage_rows)
strategy_coverage = pd.concat(
    [
        df["Cooling_Strategy_Action"].value_counts(normalize=True).rename("full_pct"),
        sample_df["Cooling_Strategy_Action"].value_counts(normalize=True).rename("sample_pct"),
    ],
    axis=1,
).fillna(0)
strategy_coverage["pct_point_difference"] = (
    strategy_coverage["sample_pct"] - strategy_coverage["full_pct"]
) * 100

display(coverage.round(3))
display(strategy_coverage.round(3))

In [ ]:
train_df, test_df = train_test_split(
    sample_df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=sample_df["Cooling_Strategy_Action"],
)

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_reg_train = train_df[regression_target]
y_reg_test = test_df[regression_target]

y_log_train = train_df[classification_target]
y_log_test = test_df[classification_target]

X_cluster_train = train_df[cluster_cols]
X_cluster_test = test_df[cluster_cols]

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Test percentage of selected sample:", round(len(test_df) / len(sample_df) * 100, 2), "%")

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
drive_service = build("drive", "v3")

def download_drive_file(filename):
    local_path = MODEL_DIR / filename
    if local_path.exists():
        return local_path

    matches = drive_service.files().list(
        q=f"name='{filename}' and '{DRIVE_MODEL_FOLDER_ID}' in parents and trashed=false",
        fields="files(id, name)",
    ).execute().get("files", [])

    if not matches:
        raise FileNotFoundError(
            f"Missing {filename} in the Drive ml_models folder. Run notebook 02 first."
        )

    request = drive_service.files().get_media(fileId=matches[0]["id"])
    with open(local_path, "wb") as f:
        downloader = MediaIoBaseDownload(f, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()

    print("Downloaded from Drive:", filename)
    return local_path

model_files = {
    "linear": "linear_regression_power_model.pkl",
    "kmeans": "kmeans_operating_cluster_model.pkl",
    "logistic": "logistic_regression_increase_chiller_model.pkl",
}

loaded = {}
for key, filename in model_files.items():
    path = download_drive_file(filename)
    with open(path, "rb") as f:
        loaded[key] = pickle.load(f)
    print("Loaded", path)

test_path = download_drive_file("test_data_for_inference.csv")
inference_df = pd.read_csv(test_path)

In [ ]:
linear_artifact = loaded["linear"]
kmeans_artifact = loaded["kmeans"]
logistic_artifact = loaded["logistic"]

inference_results = inference_df.copy()
inference_results["Predicted_Cooling_Power_kW"] = linear_artifact["model"].predict(
    inference_df[linear_artifact["features"]]
)
inference_results["Operating_Cluster"] = kmeans_artifact["model"].predict(
    inference_df[kmeans_artifact["features"]]
)
inference_results["Increase_Chiller_Probability"] = logistic_artifact["model"].predict_proba(
    inference_df[logistic_artifact["features"]]
)[:, 1]
inference_results["Predicted_Increase_Chiller"] = (
    inference_results["Increase_Chiller_Probability"] >= 0.5
).astype(int)

display(inference_results.head(20))

if "Cooling_Power_kW" in inference_results:
    print("Linear Regression inference R2:", round(r2_score(
        inference_results["Cooling_Power_kW"],
        inference_results["Predicted_Cooling_Power_kW"],
    ), 4))

if "Increase_Chiller_Label" in inference_results:
    print("Logistic Regression inference accuracy:", round(accuracy_score(
        inference_results["Increase_Chiller_Label"],
        inference_results["Predicted_Increase_Chiller"],
    ), 4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(
    inference_results["Cooling_Power_kW"],
    inference_results["Predicted_Cooling_Power_kW"],
    alpha=0.65,
)
axes[0].plot(
    [inference_results["Cooling_Power_kW"].min(), inference_results["Cooling_Power_kW"].max()],
    [inference_results["Cooling_Power_kW"].min(), inference_results["Cooling_Power_kW"].max()],
    "r--",
)
axes[0].set_title("Loaded Linear Model: Actual vs Predicted Power")
axes[0].set_xlabel("Actual Cooling Power (kW)")
axes[0].set_ylabel("Predicted Cooling Power (kW)")

cluster_plot = axes[1].scatter(
    inference_results["Server_Workload"],
    inference_results["Cooling_Power_kW"],
    c=inference_results["Operating_Cluster"],
    cmap="viridis",
    alpha=0.75,
)
axes[1].set_title("Loaded K-Means Model: Operating Clusters")
axes[1].set_xlabel("Server Workload (%)")
axes[1].set_ylabel("Cooling Power (kW)")
fig.colorbar(cluster_plot, ax=axes[1], label="Cluster")

axes[2].scatter(
    inference_results["Chiller_Usage"],
    inference_results["Increase_Chiller_Probability"],
    c=inference_results["Increase_Chiller_Label"],
    cmap="coolwarm",
    alpha=0.75,
)
axes[2].axhline(0.5, color="black", linestyle="--", linewidth=1)
axes[2].set_title("Loaded Logistic Model: Increase Chiller Probability")
axes[2].set_xlabel("Chiller Usage (%)")
axes[2].set_ylabel("Predicted Probability")

plt.tight_layout()
plt.show()

cm = confusion_matrix(
    inference_results["Increase_Chiller_Label"],
    inference_results["Predicted_Increase_Chiller"],
)
ConfusionMatrixDisplay(cm, display_labels=["Other action", "Increase Chiller"]).plot(
    cmap="Blues",
    values_format="d",
)
plt.title("Loaded Logistic Model Confusion Matrix")
plt.show()